# Agentpy

A small Python wrapper around local `codex exec`. All three public methods mutate agent state and return nothing: `llmrun()` appends an invocation record, `llmupd()` runs `self.contextUpdPrompt` verbatim and stores its answer as `self.context`, and `llmrunupd()` runs both in sequence.

The notebook kernel uses the project-local `.jupyter-venv` environment.

In [13]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from uuid import uuid4

from agentpy_codex import run_codex as llm_session
from agentpy_state import AgentState


@dataclass
class Agentpy(AgentState):
    context: str
    manifest: str
    contextUpdPrompt: str
    workdir: Path = field(default_factory=Path.cwd)
    agid: str = field(default_factory=lambda: f"ag_{uuid4().hex}")
    invocations: list[dict[str, str]] = field(default_factory=list, init=False)
    last_invocation: str | None = field(default=None, init=False)
    last_result: str | None = field(default=None, init=False)
    def llmrun(self, invocation: str) -> None:
        """Run Codex and append {invocation, answer} to self.invocations."""
        self.last_invocation = invocation
        prompt = f"""# Agent context\n{self.context}\n\n# Manifest\n{self.manifest}\n\n# Invocation\n{invocation}"""
        self.last_result = llm_session(prompt, workdir=self.workdir)
        self.invocations.append({
            "invocation": invocation,
            "answer": self.last_result,
        })
        self.save()

    def llmupd(self) -> None:
        """Replace self.context using the latest invocation record."""
        if self.last_invocation is None or self.last_result is None:
            raise ValueError("llmupd requires a pending llmrun invocation.")
        self.context = llm_session(self.contextUpdPrompt, workdir=self.workdir)
        self.last_invocation = None
        self.last_result = None
        self.save()

    def llmrunupd(self, invocation: str) -> None:
        """Run an invocation and then update self.context from its recorded answer."""
        self.llmrun(invocation)
        self.llmupd()


In [14]:
agent = Agentpy(
    context="You are a helpful coding agent.",
    manifest="Work only inside the current project and explain the result briefly.",
    contextUpdPrompt=(
        "Update the agent context with durable facts learned from the completed task. "
        "Keep it concise; preserve useful existing context and omit transient details."
    ),
)

# Uncomment one of these when you are ready to call Codex:
# agent.llmrun("Inspect this project and summarize its structure.")
# print(agent.invocations[-1])
# agent.llmupd()
# agent.llmrunupd("Inspect this project and record its main components.")
# print(agent.context)

In [17]:
agent.llmrun("Inspect this project and summarize its structure.")

In [18]:
print(agent.last_result)

This workspace is a collection of several independent projects, not one unified repository. All visible content is currently untracked at the workspace root.

- `Sapiens4Orgs/` — the largest project: an organizational-memory platform.
  - `brain/mess/`: implemented FastAPI + SQLite Slack-like messaging backend.
  - `brain/memtex/`: implemented memory ingestion, extraction/review workers, SQLite schemas, and working-memory compiler.
  - `brain/sapiens/`, `config/{adapters,guardials,router}/`: documented/planned agent, policy, integration, and routing layers.
  - `utils/frontend/`: Vite + React UI.
  - `tdd/`: Ruby-based multi-actor end-to-end scenarios, Python tests, mock server, and document/image fixtures.
  - `ops/elastic/`: Docker/Filebeat logging setup.
  - Contains its own Git repository and Python virtual environment.

- `blindly4/` — standalone Swift package and Git repository for macOS Accessibility API inspection and UI automation.
  - `Sources/blindly4/`: CLI, accessibility t

<bound method Agentpy.llmrunupd of Agentpy(context='You are a helpful coding agent.', manifest='Work only inside the current project and explain the result briefly.', contextUpdPrompt='Update the agent context with durable facts learned from the completed task. Keep it concise; preserve useful existing context and omit transient details.', workdir=PosixPath('/Users/billyjames/Documents/New project'), model=None, last_result=None)>